# Suite PAR — Parametrics

A parameter is a name a definition declares (`Schemas.OfParameter`), with a description: a symbol standing for what is
bound where the definition is referred to. Parameters are `Parametrics`', normal schemas and one extension, installed
before use ([CAP-112: parameters modeled as schema data](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-112.md)): a set of parameters (`Parametrics.OfParameters`), each named by its entry
(`Parametrics.Defines`) and for a value of a schema or an object of an object schema, declared by a definition of any
kind (`parameters`). A schema declares parameters, and a reference object that the schema governs assigns a value to
each parameter for a value and binds each parameter for an object to an object, through the adjacencies
`Parametrics.extend` gives the schema ([CAP-314: parameters bound by the data objects their schema governs](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-314)). Here: installing it, declaring and
validating parameters, snapshots writing them, extending a schema, and assigning and binding parameters.

In [ ]:
from mbse.Schemas.Framework import Builders as B, Parametrics, Plain, Schemas as S, Validators
from support import entries_in, given, raises, spec_of

<a id="PAR-01"></a>
## PAR-01 · Parametrics is installed before use: its schemas, a parameter's ends and every kind's set of parameters; installed again, it writes nothing

In [ ]:
KINDS = ("OfScalar", "OfObject", "OfUnion", "OfIntersection", "OfContainer", "OfAny", "OfRelation", "OfPattern",
         "OfExpression", "OfCollector", "OfModule")
installed = B.Store()
assert "Parametrics.OfParameters" not in installed.names()  # not installed: a store has none
assert "for_value" not in installed.Schemas.OfParameter.as_schema().adjacencies
assert "parameters" not in installed.Schemas.OfObject.as_schema().adjacencies  # the core declares none
installed.transaction(Parametrics.install)
assert [name for name in installed.names() if name.startswith("Parametrics.")] == [
    "Parametrics.OfParameters", "Parametrics.Defines", "Parametrics.ForValue", "Parametrics.ForObject",
    "Parametrics.HasParameters", "Parametrics.AssignsValue", "Parametrics.BindsObject"]
Defines, ForValue, ForObject, HasParameters, AssignsValue, BindsObject = (installed.registered(f"Parametrics.{name}") for name in (
    "Defines", "ForValue", "ForObject", "HasParameters", "AssignsValue", "BindsObject"))
assert installed.registered("Parametrics.OfParameters").ref and list(installed.registered("Parametrics.OfParameters").adjacencies) == ["parameters"]
assert list(Defines.entry.links) == ["parameters", "parameter"] and list(Defines.entry.properties) == ["name"]
assert Defines.uniques == (frozenset({"parameters", "name"}), frozenset({"parameter"}))  # keyed by name
assert ForValue.uniques == ForObject.uniques == (frozenset({"schema"}),)  # one schema per parameter, shared by many
assert HasParameters.uniques == (frozenset({"defn"}), frozenset({"params"}))  # unique in both directions
assert list(AssignsValue.entry.links) == ["object", "param"] and AssignsValue.entry.properties["value"].schema.schemas == ()  # any value
assert list(BindsObject.entry.links) == ["object", "param", "obj_ref"]  # an object binds, not a schema
OfParameter = installed.Schemas.OfParameter.as_schema()  # the extension
assert [(name, a.relation, a.me) for name, a in OfParameter.adjacencies.items()][-2:] == [
    ("for_value", ForValue, "param"), ("for_object", ForObject, "param")]
assert [(a.relation, a.me) for a in OfParameter.anonymous][-3:] == [
    (Defines, "parameter"), (AssignsValue, "param"), (BindsObject, "param")]  # what defines it, and what binds it
for kind in KINDS:
    held = getattr(installed.Schemas, kind).as_schema().adjacencies["parameters"]
    assert held.relation is HasParameters and held.me == "defn"
log = Plain.ToPlain(installed).Snapshot()
installed.transaction(Parametrics.install)  # installed again: found, nothing written
assert Plain.ToPlain(installed).Snapshot() == log
assert "Parametrics.OfParameters" not in B.Store().names()  # each store's own

<a id="PAR-02"></a>
## PAR-02 · Every kind of schema declares a set of parameters, each named by its entry, described, and for a value of a schema or an object of an object schema

In [ ]:
def declaring(txn):
    """A schema of every kind declaring a set holding the parameter n, for an int; Matrix, declaring rows and cols; and
    Tree, whose parameter is for a reference to a tree."""
    S = txn.Schemas
    return txn.uid("par.declaring/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Object": S.OfObject, "Relation": S.OfRelation, "Union": S.OfUnion, "Intersection": S.OfIntersection,
                  "Container": S.OfContainer, "Pattern": S.OfPattern, "Matrix": S.OfObject, "Tree": S.OfObject})
        .spec([
            lambda _: _.Object.name("par.Object").parameters([txn.Parametrics.OfParameters(lambda ps: ps.parameters({
                "n": lambda d: d.parameter(lambda p: p.for_value([S.OfInt]))}))]),
            lambda _: _.Relation.name("par.Relation").links([lambda t: t.name("a"), lambda t: t.name("b")]).parameters([
                txn.Parametrics.OfParameters(lambda ps: ps.parameters({
                    "n": lambda d: d.parameter(lambda p: p.for_value([S.OfInt]))}))]),
            lambda _: _.Union.name("par.Union").parameters([txn.Parametrics.OfParameters(lambda ps: ps.parameters({
                "n": lambda d: d.parameter(lambda p: p.for_value([S.OfInt]))}))]).branches([
                lambda t: t.name("a").schema([S.OfStr]), lambda t: t.name("b").schema([S.OfStr])]),
            lambda _: _.Intersection.name("par.Intersection").parameters([txn.Parametrics.OfParameters(lambda ps: ps.parameters({
                "n": lambda d: d.parameter(lambda p: p.for_value([S.OfInt]))}))]).parts([
                lambda t: t.name("a").schema([S.OfStr]), lambda t: t.name("b").schema([S.OfStr])]),
            lambda _: _.Container.name("par.Container").parameters([txn.Parametrics.OfParameters(lambda ps: ps.parameters({
                "n": lambda d: d.parameter(lambda p: p.for_value([S.OfInt]))}))]).index([S.OfInt]).element([S.OfStr]),
            lambda _: _.Pattern.name("par.Pattern").parameters([txn.Parametrics.OfParameters(lambda ps: ps.parameters({
                "n": lambda d: d.parameter(lambda p: p.for_value([S.OfInt]))}))]),
            lambda _: _.Matrix.name("Matrix").parameters([txn.Parametrics.OfParameters(lambda ps: ps.parameters({
                "rows": lambda d: d.parameter(lambda p: p.for_value([S.OfInt])),
                "cols": lambda d: d.parameter(lambda p: p.description("How many columns").for_value([S.OfInt])),
            }))]),
            lambda _: _.Tree.name("Tree").ref(True).parameters([txn.Parametrics.OfParameters(lambda ps: ps.parameters({
                "root": lambda d: d.parameter(lambda p: p.for_object([_.Tree]))}))]),
        ])]))])


declared = B.Store()
declared.transaction(Parametrics.install)
declared.transaction(declaring)
for name, kind in [("par.Object", B.OfObject), ("par.Relation", B.OfRelation), ("par.Union", B.OfUnion),
                   ("par.Intersection", B.OfIntersection), ("par.Container", B.OfContainer), ("par.Pattern", B.OfPattern)]:
    defn = declared.registered(name)
    (held,) = kind.Builder(defn, False, declared).parameters  # one set, its parameters keyed by their names
    assert [e.name.get() for e in held.params.parameters] == ["n"] and Parametrics.validate(declared, defn) == []
(held,) = B.OfObject.Builder(declared.registered("Matrix"), False, declared).parameters
rows, cols = held.params.parameters  # in order
assert (rows.name.get(), cols.name.get()) == ("rows", "cols") and isinstance(cols.parameter, S.OfParameter.Data)
assert (rows.parameter.description, cols.parameter.description) == (None, "How many columns")
assert S.OfParameter.Data("w", "Its width") == S.OfParameter.Data("w", "Its width") != S.OfParameter.Data("w")  # by what it holds
assert [e.schema.name.get() for e in B.OfParameter.Builder(cols.parameter, False, declared).for_value] == ["Schemas.OfInt"]
(held,) = B.OfObject.Builder(declared.registered("Tree"), False, declared).parameters
(root,) = held.params.parameters
ends = B.OfParameter.Builder(root.parameter, False, declared)
assert [e.schema.name.get() for e in ends.for_object] == ["Tree"] and list(ends.for_value) == []  # for an object
log = Plain.ToPlain(declared).Snapshot()
declared.transaction(declaring)  # run again: found, nothing written
assert Plain.ToPlain(declared).Snapshot() == log
with declared.transaction():  # imperatively, the set built where the definition links it
    Vector = declared.Schemas.OfObject(lambda o: o.name("Vector").parameters([declared.Parametrics.OfParameters(
        lambda ps: ps.parameters({"n": lambda d: d.parameter(lambda p: p.for_value([declared.Schemas.OfInt]))}))]))
(held,) = B.OfObject.Builder(Vector, False, declared).parameters
assert [e.name.get() for e in held.params.parameters] == ["n"] and Parametrics.validate(declared, Vector) == []
assert Parametrics.validate(declared, declared.Schemas.OfInt.as_schema()) == []  # none declared: none to check

<a id="PAR-03"></a>
## PAR-03 · Validation checks a set's entries: reserved names, descriptions, a parameter for one of a value and an object, a pattern's names, and one set to one definition

In [ ]:
def flawed(txn):
    """A pattern whose set has a reserved name, a parameter for neither a value nor an object, one for both, and names
    its symbol and its variable have; and Shared and Twice, which share one set."""
    S = txn.Schemas
    return txn.uid("par.flawed/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Pattern": S.OfPattern, "Node": S.OfObject})
        .spec([
            lambda _: _.Node.name("par.Node").ref(True),
            lambda _: _.Pattern.name("par.Flawed").symbols({"x": _.Node}).variables({"w": S.OfInt}).parameters([
                txn.Parametrics.OfParameters(lambda ps: ps.parameters({
                    "$n": lambda d: d.parameter(lambda p: p.for_value([S.OfInt])),
                    "x": lambda d: d.parameter(lambda p: p.for_object([_.Node])),
                    "w": lambda d: d.parameter(lambda p: p),  # for neither
                    "b": lambda d: d.parameter(lambda p: p.for_value([S.OfInt]).for_object([_.Node])),  # for both
                }))]),
        ])]))])


checked = B.Store()
checked.transaction(Parametrics.install)
checked.transaction(flawed)
Flawed = checked.registered("par.Flawed")
assert Parametrics.validate(checked, Flawed) == [
    "name '$n' is reserved: names starting with '$' belong to the wire format",
    "parameter 'w' is for neither a value nor an object: a parameter is for one",
    "parameter 'b' is for both a value and an object: a parameter is for one",
    "names used as both symbol and parameter: ['x']",
    "names used as both variable and parameter: ['w']"]
assert Flawed.validate() == []  # a kind's own validate() does not read the extension's entries
(held,) = B.OfPattern.Builder(Flawed, False, checked).parameters
dollar = next(iter(held.params.parameters)).parameter
dollar.description = True  # data the builders refuse to build
assert Parametrics.validate(checked, Flawed)[1] == "parameter '$n': a description is text, got bool"
with raises(TypeError, match="property 'description' holds str, got bool"):
    with checked.transaction():
        checked.Schemas.OfParameter(lambda p: p.description(True))
with checked.transaction():  # one set linked by two definitions, and a definition linking two sets
    shared = checked.Parametrics.OfParameters(lambda ps: ps.parameters({
        "n": lambda d: d.parameter(lambda p: p.for_value([checked.Schemas.OfInt]))}))
    other = checked.Parametrics.OfParameters(lambda ps: ps)
    One = checked.Schemas.OfObject(lambda o: o.name("par.One").parameters([shared]))
    Two = checked.Schemas.OfObject(lambda o: o.name("par.Two").parameters([shared, other]))
assert Parametrics.validate(checked, One) == ["its set of parameters belongs to 2 definitions: a set belongs to one"]
assert Parametrics.validate(checked, Two) == ["it has 2 sets of parameters: a definition has one",
                                              "its set of parameters belongs to 2 definitions: a set belongs to one"]

<a id="PAR-04"></a>
## PAR-04 · Snapshots write a definition's set as the extension's entry and read it back; a store without Parametrics ignores what needs it

In [ ]:
def grid(txn):
    """Grid, declaring rows and cols, each for an int, its cells a container of floats."""
    S = txn.Schemas
    return txn.uid("par.grid/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Grid": S.OfObject})
        .spec([
            lambda _: _.Grid.name("Grid").parameters([txn.Parametrics.OfParameters(lambda ps: ps.parameters({
                "rows": lambda d: d.parameter(lambda p: p.for_value([S.OfInt])),
                "cols": lambda d: d.parameter(lambda p: p.description("Its width").for_value([S.OfInt])),
            }))]).properties({"cells": S.OfContainer(lambda c: c.index([S.OfInt]).element([S.OfFloat]))}),
        ])]))])


written = B.Store()
written.transaction(Parametrics.install)
written.transaction(grid)
log = Plain.ToPlain(written).Snapshot()
(entry,) = entries_in(spec_of(log, "Grid", 1), "parameters")  # the extension's entry, with the data: written once
assert entry == {"params": {"reference": {"symbol": "0"}}} and entries_in(spec_of(log, "Grid"), "parameters") == []
parameters = entries_in(spec_of(log, "0", 1), "parameters")  # the set's, each defining the parameter its entry links
assert [e["name"] for e in parameters] == ["cols", "rows"]  # data's keyed entries, written in the order of their keys
assert [given(e["parameter"]["def"]["mutations"]) for e in parameters] == [{"description": "Its width"}, {}]
back = B.Store()
back.transaction(Parametrics.install)  # a store with Parametrics installed
Plain.FromPlain(back).Snapshot(log)
assert Plain.ToPlain(back).Snapshot() == log  # read back, written again: the same
(held,) = B.OfObject.Builder(back.registered("Grid"), False, back).parameters
assert [e.name.get() for e in held.params.parameters] == ["cols", "rows"]
assert Parametrics.validate(back, back.registered("Grid")) == [] and back.registered("Grid") is not written.registered("Grid")
back.transaction(grid)  # its install, run again: found, nothing written
assert Plain.ToPlain(back).Snapshot() == log
bare, ignored = B.Store(), []  # a store without it reads it as a snapshot made with extensions it lacks
Plain.FromPlain(bare).Snapshot(log, ignored)
assert [i["reason"] for i in ignored] == ["Schemas.OfParameter has no adjacency 'for_value'"] * 2 + [
    "Schemas.OfObject has no adjacency 'parameters'"]
assert list(bare.registered("Grid").properties) == ["cells"] and "parameters" not in bare.Schemas.OfObject.as_schema().adjacencies

<a id="PAR-05"></a>
## PAR-05 · `extend` gives a reference object schema the adjacencies through which its objects assign and bind; it refuses a store without Parametrics, a value object schema and a member's name taken; run again, it writes nothing

In [ ]:
def hardware(txn):
    """hw.Bus, a reference object declaring width, for an int, and host, for a hw.Chip; hw.Chip and hw.Spare, reference
    objects whose schemas declare nothing; hw.Pin, a value object; hw.Taken and hw.Linked, which declare a member of
    each name extend gives."""
    S = txn.Schemas
    return txn.uid("par.hardware/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Bus": S.OfObject, "Chip": S.OfObject, "Spare": S.OfObject, "Pin": S.OfObject, "Taken": S.OfObject,
                  "Linked": S.OfObject, "Wires": S.OfRelation})
        .spec([
            lambda _: _.Chip.name("hw.Chip").ref(True),  # opts into nothing: any reference object may be bound
            lambda _: _.Spare.name("hw.Spare").ref(True),
            lambda _: _.Bus.name("hw.Bus").ref(True).properties({"label": S.OfStr}).parameters([
                txn.Parametrics.OfParameters(lambda ps: ps.parameters({
                    "width": lambda d: d.parameter(lambda p: p.for_value([S.OfInt])),
                    "host": lambda d: d.parameter(lambda p: p.for_object([_.Chip])),
                }))]),
            lambda _: _.Pin.name("hw.Pin").properties({"number": S.OfInt}),
            lambda _: _.Wires.name("hw.Wires").links([lambda t: t.name("from"), lambda t: t.name("to")]),
            lambda _: _.Taken.name("hw.Taken").ref(True).properties({"assigned_values": S.OfStr}),
            lambda _: _.Linked.name("hw.Linked").ref(True).adjacencies({
                "bound_objects": lambda a: a.relation([_.Wires]).me("from")}),
        ])]))])


bare = B.Store()
with raises(LookupError, match="Parametrics.extend: the store has no Parametrics installed; install it first, "
                                "store.transaction(Parametrics.install)"):
    bare.transaction(lambda t: Parametrics.extend(t, [t.Schemas.OfObject]))
hw = B.Store()
hw.transaction(Parametrics.install)
hw.transaction(hardware)
Bus, AssignsValue, BindsObject = (hw.registered(name) for name in (
    "hw.Bus", "Parametrics.AssignsValue", "Parametrics.BindsObject"))
assert "assigned_values" not in Bus.adjacencies  # not extended: its objects assign and bind nothing
hw.transaction(lambda t: Parametrics.extend(t, [t.hw.Bus]))
assert [(name, a.relation, a.me) for name, a in Bus.adjacencies.items()] == [
    ("assigned_values", AssignsValue, "object"), ("bound_objects", BindsObject, "object")]
log = Plain.ToPlain(hw).Snapshot()
hw.transaction(lambda t: Parametrics.extend(t, [t.hw.Bus, hw.registered("hw.Bus")]))  # run again: nothing written
assert Plain.ToPlain(hw).Snapshot() == log
with raises(TypeError, match="Parametrics.extend takes reference object schemas: 'hw.Pin' is a value object schema, "
                             "whose objects are never linked, so that none assigns or binds a parameter"):
    hw.transaction(lambda t: Parametrics.extend(t, [t.hw.Pin]))
with raises(TypeError, match="Parametrics.extend takes reference object schemas: 'Schemas.OfInt' is not one"):
    hw.transaction(lambda t: Parametrics.extend(t, [t.Schemas.OfInt]))
with raises(TypeError, match="Parametrics.extend takes reference object schemas: 'hw.Wires' is not one"):
    hw.transaction(lambda t: Parametrics.extend(t, [t.hw.Wires]))
with raises(TypeError, match="Parametrics.extend takes reference object schemas: 'nothing' is not one"):
    hw.transaction(lambda t: Parametrics.extend(t, ["nothing"]))
with raises(TypeError, match="Parametrics.extend: 'hw.Taken' already declares a member 'assigned_values'"):
    hw.transaction(lambda t: Parametrics.extend(t, [t.hw.Taken]))
with raises(TypeError, match="Parametrics.extend: 'hw.Linked' already declares a member 'bound_objects'"):
    hw.transaction(lambda t: Parametrics.extend(t, [t.hw.Linked]))
assert Plain.ToPlain(hw).Snapshot() == log  # each refused whole: nothing written


<a id="PAR-06"></a>
## PAR-06 · An object assigns a value to a parameter its schema declares, and binds one to an object of any schema, which opts into nothing; a parameter it neither assigns nor binds stays unbound

In [ ]:
with hw.transaction():
    (held,) = B.OfObject.Builder(Bus, False, hw).parameters
    width, host = (e.parameter for e in held.params.parameters)
    chip = hw.hw.Chip(lambda c: c)
    bus = hw.hw.Bus(lambda b: b.label("data")
                    .assigned_values([lambda e: e.param(width).value(8)])
                    .bound_objects(lambda a: a.insert(lambda e: e.param(host).obj_ref(chip))))
    (value,), (obj,) = bus.assigned_values, bus.bound_objects
    assert value.param is width and value.value.get() == 8  # its value an entry property
    assert obj.param is host and obj.obj_ref == chip  # its object a third link
    narrow = hw.hw.Bus(lambda b: b.label("narrow").assigned_values([lambda e: e.param(width).value(1)]))
    assert list(narrow.bound_objects) == []  # host left unbound: it stays unbound
    loose = hw.hw.Bus(lambda b: b.label("loose"))
    assert list(loose.assigned_values) == list(loose.bound_objects) == []  # neither is required
    assert Parametrics.validate(hw, bus) == Parametrics.validate(hw, narrow) == Parametrics.validate(hw, loose) == []
    chip_schema, spare_schema = hw.registered("hw.Chip"), hw.registered("hw.Spare")
    assert list(chip_schema.adjacencies) == list(spare_schema.adjacencies) == chip_schema.anonymous == []
    spare = hw.hw.Spare(lambda s: s)
    other = hw.hw.Bus(lambda b: b.label("other").bound_objects(lambda a: a.insert(lambda e: e.param(host).obj_ref(spare))))
    validate = Validators.Validate(hw)
    assert validate(Bus, bus) == []  # obj_ref links a schema that declares nothing
    assert validate(Bus, other) == [  # but a spare is not of host's schema (PAR-13)
        "hw.Bus#0.bound_objects[0].obj_ref: a 'hw.Spare' is bound to a parameter for an object of 'hw.Chip'"]
    linked = hw.hw.Linked(lambda x: x.bound_objects(lambda a: a.insert(lambda e: e.to(spare))))
    assert validate(hw.registered("hw.Linked"), linked) == [  # another relation's link still asks for an adjacency
        "hw.Linked#0.bound_objects[0].to: a 'hw.Spare' cannot fill link 'to'; its schema declares no adjacency, named or "
        "anonymous, to this relation via 'to'"]
    board = hw.hw.Bus(lambda b: b.label("board"))
    n = hw.References.OfParameter(lambda p: p.name("n"))  # a parameter reference in the place of the parameter
    hw.builder("hw.Bus", board).assigned_values([lambda e: e.param(n).value(3)]).update()  # what it assigns is unknown here
    assert Parametrics.validate(hw, board) == []


<a id="PAR-07"></a>
## PAR-07 · An assignment or a binding of a parameter its object's schema does not declare refuses its transaction when it commits; validation reports one already held, and a snapshot reads it back as written

In [ ]:
def wiring(txn):
    """hw.Wire, declaring depth, and hw.Top, a singleton declaring n and m."""
    S = txn.Schemas
    return txn.uid("par.wiring/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Wire": S.OfObject, "Top": S.OfObject})
        .spec([
            lambda _: _.Wire.name("hw.Wire").ref(True).parameters([txn.Parametrics.OfParameters(lambda ps: ps.parameters({
                "depth": lambda d: d.parameter(lambda p: p.for_value([S.OfInt]))}))]),
            lambda _: _.Top.name("hw.Top").ref(True).singleton("hw.Top").parameters([
                txn.Parametrics.OfParameters(lambda ps: ps.parameters({
                    "n": lambda d: d.parameter(lambda p: p.for_value([S.OfInt])),
                    "m": lambda d: d.parameter(lambda p: p.for_value([S.OfInt]))}))]),
        ])]))])


hw.transaction(wiring)
hw.transaction(lambda t: Parametrics.extend(t, [t.hw.Wire, t.hw.Top]))
log = Plain.ToPlain(hw).Snapshot()
with hw.transaction():
    (held,) = B.OfObject.Builder(hw.registered("hw.Wire"), False, hw).parameters
    (depth,) = (e.parameter for e in held.params.parameters)
    free = hw.Schemas.OfParameter(lambda p: p.for_value([hw.Schemas.OfInt]))  # in no set
with raises(ValueError, match="an entry of 'Parametrics.AssignsValue' assigns a value to parameter 'depth', which 'hw.Bus' "
                              "does not declare: an object assigns and binds only the parameters its schema declares"):
    with hw.transaction():
        hw.hw.Bus(lambda b: b.assigned_values([lambda e: e.param(depth).value(8)]))  # a hw.Wire's: written, and
                                                                                  # refused at commit, as a unique clause is
with raises(ValueError, match="an entry of 'Parametrics.BindsObject' binds a parameter no set of parameters defines to "
                              "an object, which 'hw.Bus' does not declare"):
    with hw.transaction():
        hw.hw.Bus(lambda b: b.bound_objects(lambda a: a.insert(lambda e: e.param(free).obj_ref(hw.hw.Chip(lambda c: c)))))
with raises(ValueError, match="link 'param' is not set"):  # an assignment names its parameter
    with hw.transaction():
        hw.hw.Bus(lambda b: b.assigned_values(lambda a: a.insert(lambda e: e.value(8))))
assert Plain.ToPlain(hw).Snapshot() == log  # each refused whole
with hw.transaction():
    (held,) = B.OfObject.Builder(hw.registered("hw.Top"), False, hw).parameters
    n, m = (e.parameter for e in held.params.parameters)
    hw.builder("hw.Top", hw.singleton("hw.Top")).assigned_values([lambda e: e.param(n).value(3)]).update()
with hw.transaction():  # an object that assigns one already assigns another: the assignment inserted is checked
    hw.builder("hw.Top", hw.singleton("hw.Top")).assigned_values(lambda a: a.insert(lambda e: e.param(m).value(4))).update()
    assert [e.value.get() for e in hw.singleton("hw.Top").assigned_values] == [3, 4]
with hw.transaction():  # the set's declarations removed: the assignments held stay
    (held,) = B.OfObject.Builder(hw.registered("hw.Top"), False, hw).parameters
    hw.builder("Parametrics.OfParameters", held.params._target()).parameters(lambda a: a.clear()).update()
with hw.transaction():  # n and m no longer declared: the assignments held are reported, not refused
    assert Parametrics.validate(hw, hw.singleton("hw.Top")) == [
        "an entry of 'Parametrics.AssignsValue' assigns a value to a parameter no set of parameters defines, which "
        "'hw.Top' does not declare: an object assigns and binds only the parameters its schema declares"] * 2
written = Plain.ToPlain(hw).Snapshot()  # the assignments, written as any entries are
back = B.Store()
back.transaction(Parametrics.install)
Plain.FromPlain(back).Snapshot(written)  # read back as written: not refused
assert Plain.ToPlain(back).Snapshot() == written
with back.transaction():
    assert [e.value.get() for e in back.singleton("hw.Top").assigned_values] == [3, 4]
    assert len(Parametrics.validate(back, back.singleton("hw.Top"))) == 2


<a id="PAR-08"></a>
## PAR-08 · A schema neither assigns nor binds its parameters: no kind of schema has an adjacency to an assignment or a binding, each links an `object`, and extending a schema of schemas is refused

In [ ]:
for kind in KINDS:  # each kind holds its set of parameters, and no assignment or binding
    schema = getattr(hw.Schemas, kind).as_schema()
    assert all(a.relation not in (AssignsValue, BindsObject)
               for a in (*schema.adjacencies.values(), *schema.anonymous))
assert [link for link in AssignsValue.entry.links] == ["object", "param"] and "defn" not in BindsObject.entry.links
with raises(TypeError, match="Parametrics.extend: 'Schemas.OfObject' is a schema of schemas, whose objects are "
                             "schemas: a schema declares its parameters and neither assigns nor binds them"):
    hw.transaction(lambda t: Parametrics.extend(t, [t.Schemas.OfObject]))
assert "assigned_values" not in hw.Schemas.OfObject.as_schema().adjacencies


<a id="PAR-09"></a>
## PAR-09 · A parameter context maps a parameter's name to the value assigned to it or the object bound to it; `assign` and `bind` give a sub-context, which maps the name anew and hides the context's own mapping of it, the context unchanged

In [ ]:
def boards(txn):
    """pcb.Board, a reference object declaring n, for an int, and host, for a pcb.Chip, its size an int, its cells ints,
    its tag an int as a value of any schema, and the chips it feeds; pcb.Chip, which a board may feed; pcb.Spare, which
    none may."""
    S = txn.Schemas
    return txn.uid("par.boards/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Board": S.OfObject, "Chip": S.OfObject, "Spare": S.OfObject, "Feeds": S.OfRelation})
        .spec([
            lambda _: _.Feeds.name("pcb.Feeds").links([lambda t: t.name("board"), lambda t: t.name("chip")]),
            lambda _: _.Chip.name("pcb.Chip").ref(True).adjacencies([lambda a: a.relation([_.Feeds]).me("chip")]),
            lambda _: _.Spare.name("pcb.Spare").ref(True),
            lambda _: _.Board.name("pcb.Board").ref(True).properties({
                "size": S.OfInt, "cells": S.OfContainer(lambda c: c.index([S.OfInt]).element([S.OfInt])),
                "tag": S.OfAny(lambda a: a.schemas([S.OfInt]))}).adjacencies({
                "feeds": lambda a: a.relation([_.Feeds]).me("board")}).parameters([
                txn.Parametrics.OfParameters(lambda ps: ps.parameters({
                    "n": lambda d: d.parameter(lambda p: p.for_value([S.OfInt])),
                    "host": lambda d: d.parameter(lambda p: p.for_object([_.Chip])),
                }))]),
        ])]))])


pcb = B.Store()
pcb.transaction(Parametrics.install)
pcb.transaction(boards)
pcb.transaction(lambda t: Parametrics.extend(t, [t.pcb.Board]))
Board = pcb.registered("pcb.Board")
n, host, q = (pcb.References.OfParameter(lambda p, name=name: p.name(name)) for name in ("n", "host", "q"))
validate, check = Validators.Validate(pcb), Validators.Check(pcb)
empty = Parametrics.Context()
assert empty.assigned("n") is None and empty.bound("n") is None  # it maps no name
target = object()  # an object or a symbol: what is bound is the caller's
assigned = empty.assign("n", 3)
rebound = assigned.bind("n", target)
assert assigned.assigned("n") == 3 and assigned.bound("n") is None
assert rebound.bound("n") is target and rebound.assigned("n") is None  # mapped anew: the outer assignment hidden
assert empty.assigned("n") is None and assigned.assigned("n") == 3  # each context unchanged
assert rebound.assign("m", 4).bound("n") is target and rebound.assign("m", 4).assigned("m") == 4  # other names seen

<a id="PAR-10"></a>
## PAR-10 · Validation checks a parameter reference in the sub-context that the object holding it makes: valid where its parameter is assigned a value, or bound to an object, of the schema its place expects; invalid where it is neither, or is the wrong kind of thing

In [ ]:
with pcb.transaction():
    (held,) = B.OfObject.Builder(Board, False, pcb).parameters
    n_param, host_param = (e.parameter for e in held.params.parameters)
    chip, spare = pcb.pcb.Chip(lambda c: c), pcb.pcb.Spare(lambda s: s)
    good = pcb.pcb.Board(lambda b: b.size(n).cells({1: n, 2: 5}).tag(n).feeds([lambda e: e.chip(host)])
                         .assigned_values([lambda e: e.param(n_param).value(3)])
                         .bound_objects(lambda a: a.insert(lambda e: e.param(host_param).obj_ref(chip))))
    assert validate(Board, good) == [] and check(Board, good).holds is True  # each valid: its object assigns or binds it
    bare = pcb.pcb.Board(lambda b: b.size(n).cells({1: n}).feeds([lambda e: e.chip(host)]))
    assert validate(Board, bare) == [  # in an empty context: neither assigned nor bound
        "pcb.Board#0.size: parameter 'n' is neither assigned nor bound in its context",
        "pcb.Board#0.cells[1]: parameter 'n' is neither assigned nor bound in its context",
        "pcb.Board#0.feeds[0].chip: parameter 'host' is neither assigned nor bound in its context"]
    swapped = pcb.pcb.Board(lambda b: b.size(host).feeds([lambda e: e.chip(n)])
                            .assigned_values([lambda e: e.param(n_param).value(3)])
                            .bound_objects(lambda a: a.insert(lambda e: e.param(host_param).obj_ref(chip))))
    assert validate(Board, swapped) == [  # an object where a value is expected, and a value where an object is
        "pcb.Board#0.size: parameter 'host' is bound to an object, where a value is expected",
        "pcb.Board#0.feeds[0].chip: parameter 'n' is assigned a value, where an object is expected"]
    assert check(Board, swapped).holds is False

<a id="PAR-11"></a>
## PAR-11 · Validation takes a parameter context the caller gives, at every entry point: a reference valid where the context assigns or binds its parameter, of the right schema, and invalid where it maps it to something of another schema; an object's own assignments and bindings hide the caller's

In [ ]:
with pcb.transaction():
    chip, spare = pcb.pcb.Chip(lambda c: c), pcb.pcb.Spare(lambda s: s)
    good, bare = (pcb.pcb.Board(lambda b: b.size(n).cells({1: n}).tag(n).feeds([lambda e: e.chip(host)])) for _ in "ab")
    pcb.builder("pcb.Board", good).assigned_values([lambda e: e.param(n_param).value(3)]).bound_objects(
        lambda a: a.insert(lambda e: e.param(host_param).obj_ref(chip))).update()
    caller = Parametrics.Context().assign("n", 3).bind("host", chip)
    assert validate(Board, bare, caller) == validate.OfObject(Board, bare, caller) == validate.Reachable(Board, bare, caller) == []
    assert check(Board, bare, caller).holds is check.OfObject(Board, bare, caller).holds is check.Reachable(Board, bare, caller).holds is True
    wrong = Parametrics.Context().assign("n", "three").bind("host", spare)
    assert validate(Board, bare, wrong) == [  # the wrong schema: a str for an int, a spare where a chip is fed
        "pcb.Board#0.size: parameter 'n' is assigned a value of another schema than its place expects",
        "pcb.Board#0.cells[1]: parameter 'n' is assigned a value of another schema than its place expects",
        "pcb.Board#0.tag: parameter 'n' is assigned a value of another schema than its place expects",
        "pcb.Board#0.feeds[0].chip: parameter 'host' is bound to an object of another schema: a 'pcb.Spare' cannot fill "
        "link 'chip'; its schema declares no adjacency, named or anonymous, to this relation via 'chip'"]
    assert validate(Board, good, wrong) == []  # the object's own assignment and binding hide the caller's
    assert validate(Board, bare, Parametrics.Context().assign("n", pcb.pcb.Chip(lambda c: c))) == [
        "pcb.Board#0.size: parameter 'n' is assigned a value of another schema than its place expects",
        "pcb.Board#0.cells[1]: parameter 'n' is assigned a value of another schema than its place expects",
        "pcb.Board#0.tag: parameter 'n' is assigned a value of another schema than its place expects",
        "pcb.Board#0.feeds[0].chip: parameter 'host' is neither assigned nor bound in its context"]
Int = pcb.Schemas.OfInt
assert validate(Int, n, caller) == validate.OfScalar(Int, n, caller) == [] and check.OfScalar(Int, n, caller).holds is True
assert validate.OfScalar(Int, n) == ["the value: parameter 'n' is neither assigned nor bound in its context"]
assert check(Int, n, wrong).problems == ["the value: parameter 'n' is assigned a value of another schema than its place expects"]
assert validate(Int, 3, wrong) == [] and validate(Int, "3") == ["expected int, got str"]  # a scalar: as without a context

<a id="PAR-12"></a>
## PAR-12 · A sub-context follows validation's descent, through each object it checks, and hides an outer name: an object's assignments and bindings map its parameters for its properties and other entries, while its own are checked in the context around it

In [ ]:
def chain(txn):
    """pcb.Rack, a singleton declaring n, for an int, owning cards by key, each in a slot, an int; pcb.Card, declaring n,
    for an int, its size an int."""
    S = txn.Schemas
    return txn.uid("par.chain/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Rack": S.OfObject, "Holds": S.OfRelation, "Card": S.OfObject})
        .spec([
            lambda _: _.Holds.name("pcb.Holds").links([lambda t: t.name("rack"), lambda t: t.name("card")]).properties({
                "key": S.OfStr, "slot": S.OfInt}).uniques([["rack", "key"], ["card"]]),
            lambda _: _.Card.name("pcb.Card").ref(True).properties({"size": S.OfInt}).adjacencies({
                "rack": lambda a: a.relation([_.Holds]).me("card").owner(True)}).parameters([
                txn.Parametrics.OfParameters(lambda ps: ps.parameters({
                    "n": lambda d: d.parameter(lambda p: p.for_value([S.OfInt]))}))]),
            lambda _: _.Rack.name("pcb.Rack").ref(True).singleton("pcb.TheRack").adjacencies({
                "cards": lambda a: a.relation([_.Holds]).me("rack")}).parameters([
                txn.Parametrics.OfParameters(lambda ps: ps.parameters({
                    "n": lambda d: d.parameter(lambda p: p.for_value([S.OfInt]))}))]),
        ])]))])


pcb.transaction(chain)
pcb.transaction(lambda t: Parametrics.extend(t, [t.pcb.Rack, t.pcb.Card]))
Rack, Card = pcb.registered("pcb.Rack"), pcb.registered("pcb.Card")
rack_n, card_n = (next(iter(B.OfObject.Builder(schema, False, pcb).parameters)).params.parameters[0].parameter
                  for schema in (Rack, Card))
with pcb.transaction():
    rack = pcb.singleton("pcb.TheRack")
    pcb.pcb.Rack(rack, lambda r: r.assigned_values([lambda e: e.param(rack_n).value("four")]).cards([
        lambda e: e.key("a").card(lambda c: c.size(n).assigned_values([lambda e: e.param(card_n).value(4)])).slot(n),
        lambda e: e.key("b").card(lambda c: c.size(n)).slot(n)]))
    assert validate.Reachable(Rack, rack, Parametrics.Context().assign("n", 2)) == [
        "pcb.Rack#0.cards[0].slot: parameter 'n' is assigned a value of another schema than its place expects",
        "pcb.Rack#0.cards[1].slot: parameter 'n' is assigned a value of another schema than its place expects",
        "pcb.Rack#0.assigned_values[0].value: a value of str is assigned to a parameter for a value of int",
    ]  # the rack's "four" hides the caller's 2 for its entries; card a assigns its own 4; card b sees the caller's 2
    pcb.pcb.Rack(rack, lambda r: r.assigned_values(lambda a: a.clear().insert(lambda e: e.param(rack_n).value(4))))
    assert validate.Reachable(Rack, rack) == [  # what the rack assigns, its entries see; the cards it reaches do not,
        "pcb.Card#2.size: parameter 'n' is neither assigned nor bound in its context",  # and an entry seen from card b
        "pcb.Card#2.rack[0].slot: parameter 'n' is neither assigned nor bound in its context"]  # is checked in b's
    loop = pcb.pcb.Board(lambda b: b.assigned_values([lambda e: e.param(n_param).value(5)])
                         .bound_objects(lambda a: a.insert(lambda e: e.param(host_param).obj_ref(host))))
    assert validate(Board, loop) == [  # its own binding resolves host in the context around it, not through itself
        "pcb.Board#0.bound_objects[0].obj_ref: parameter 'host' is neither assigned nor bound in its context"]

<a id="PAR-13"></a>
## PAR-13 · Validation checks a parameter assignment's value against its parameter's `ForValue` schema, and a parameter binding's object against its `ForObject` schema: a value of another schema, an object of another schema, a value assigned to a parameter for an object and an object bound to one for a value are each invalid

In [ ]:
with pcb.transaction():
    chip, spare = pcb.pcb.Chip(lambda c: c), pcb.pcb.Spare(lambda s: s)
    mixed = pcb.pcb.Board(lambda b: b
                          .assigned_values([lambda e: e.param(n_param).value("three"), lambda e: e.param(host_param).value(3),
                                            lambda e: e.param(n_param).value(n)])
                          .bound_objects(lambda a: a.insert(lambda e: e.param(host_param).obj_ref(spare))
                                         .insert(lambda e: e.param(n_param).obj_ref(chip))))
    assert validate(Board, mixed) == [
        "pcb.Board#0.assigned_values[0].value: a value of str is assigned to a parameter for a value of int",
        "pcb.Board#0.assigned_values[1]: a value is assigned to a parameter for an object",
        "pcb.Board#0.assigned_values[2].value: a value of 'References.OfParameter' is assigned to a parameter for a value "
        "of int",  # a parameter reference where a value of any schema is declared: a value of its own schema
        "pcb.Board#0.bound_objects[0].obj_ref: a 'pcb.Spare' is bound to a parameter for an object of 'pcb.Chip'",
        "pcb.Board#0.bound_objects[1]: an object is bound to a parameter for a value"]
    unnamed = pcb.pcb.Board(lambda b: b.assigned_values([lambda e: e.param(q).value("any")]))
    assert validate(Board, unnamed) == [  # a parameter reference in the parameter's place: its schema is not known here
        "pcb.Board#0.assigned_values[0].param: parameter 'q' is neither assigned nor bound in its context"]
    assert validate(Board, unnamed, Parametrics.Context().bind("q", n_param)) == []
with hw.transaction():  # parameters no set of parameters defines any more (PAR-07): nothing maps them, nor checks them
    assert Validators.Validate(hw)(hw.registered("hw.Top"), hw.singleton("hw.Top")) == []

<a id="PAR-14"></a>
## PAR-14 · A parameter binding whose object is a parameter reference resolves it in the context around its object: what that context maps it to maps its parameter, checked against the parameter's `ForObject` schema; where the context maps it to nothing, the reference is invalid, and a reference to the parameter at a link is unknown

In [ ]:
with pcb.transaction():
    chip, spare = pcb.pcb.Chip(lambda c: c), pcb.pcb.Spare(lambda s: s)
    relay = pcb.pcb.Board(lambda b: b.feeds([lambda e: e.chip(host)])
                          .bound_objects(lambda a: a.insert(lambda e: e.param(host_param).obj_ref(q))))
    assert validate(Board, relay, Parametrics.Context().bind("q", chip)) == []  # host bound to the chip q is bound to
    assert validate(Board, relay, Parametrics.Context().bind("q", spare)) == [
        "pcb.Board#0.feeds[0].chip: parameter 'host' is bound to an object of another schema: a 'pcb.Spare' cannot fill "
        "link 'chip'; its schema declares no adjacency, named or anonymous, to this relation via 'chip'",
        "pcb.Board#0.bound_objects[0].obj_ref: a 'pcb.Spare' is bound to a parameter for an object of 'pcb.Chip'"]
    assert validate(Board, relay, Parametrics.Context().assign("q", 7)) == [
        "pcb.Board#0.feeds[0].chip: parameter 'host' is assigned a value, where an object is expected",
        "pcb.Board#0.bound_objects[0].obj_ref: parameter 'q' is assigned a value, where an object is expected"]
    outcome = check(Board, relay)  # q mapped to nothing: the binding invalid, and what host stands for unknown
    assert outcome.problems == ["pcb.Board#0.bound_objects[0].obj_ref: parameter 'q' is neither assigned nor bound in "
                                "its context"]
    assert outcome.unknowns == ["pcb.Board#0.feeds[0].chip: parameter 'host' is bound to parameter 'q', which its context "
                                "neither assigns nor binds"] and outcome.holds is False
    pcb.builder("pcb.Board", relay).size(host).update()
    assert validate(Board, relay)[0] == "pcb.Board#0.size: parameter 'host' is bound to an object, where a value is expected"